In [3]:
import tensorflow as tf
import tensorflow_data_validation as tfdv
import pandas as pd
import numpy as np
from tensorflow_metadata.proto.v0 import schema_pb2

print('Tf version: ', tf.__version__)
print('TFDV version: ', tfdv.__version__)

Tf version:  2.17.0
TFDV version:  1.14.0


In [28]:
# -------------------------
# 📂 Load raw data
# -------------------------
raw_df = pd.read_csv('dataset/laptop_data_1M_v2_CLEANED.csv')
print(f"Loaded {len(raw_df):,} rows.")

# -------------------------
# 🔀 Split into train, eval, test
# -------------------------
train_df = raw_df.sample(frac=0.8, random_state=42)
remaining_df = raw_df.drop(train_df.index)
eval_df = remaining_df.sample(frac=0.5, random_state=42)  # 10% of total
test_df = remaining_df.drop(eval_df.index)                 # 10% of total

print(
    f"Train: {len(train_df)}, Eval: {len(eval_df)}, Test: {len(test_df)}"
)


Loaded 1,000,000 rows.
Train: 800000, Eval: 100000, Test: 100000


In [29]:
# Start with an empty schema
schema = schema_pb2.Schema()

# Define Price as float with value range
price_feature = schema.feature.add()
price_feature.name = 'Price'
price_feature.type = schema_pb2.FeatureType.FLOAT
price_domain = price_feature.float_domain
price_domain.min = 10000.0
price_domain.max = 300000.0

In [30]:
# Define Company as string
company_feature = schema.feature.add()
company_feature.name = 'Company'
company_feature.type = schema_pb2.FeatureType.BYTES

In [31]:
# Define TypeName as string
typename_feature = schema.feature.add()
typename_feature.name = 'TypeName'
typename_feature.type = schema_pb2.FeatureType.BYTES

In [32]:
# Define Inches as float
inches_feature = schema.feature.add()
inches_feature.name = 'Inches'
inches_feature.type = schema_pb2.FeatureType.FLOAT
inches_domain = inches_feature.float_domain
inches_domain.min = 10.0
inches_domain.max = 32.0

In [33]:
# Define ScreenResolution as string
screen_resolution_feature = schema.feature.add()
screen_resolution_feature.name = 'ScreenResolution'
screen_resolution_feature.type = schema_pb2.FeatureType.BYTES

In [34]:
# Define Ram as float
ram_feature = schema.feature.add()
ram_feature.name = 'Ram'
ram_feature.type = schema_pb2.FeatureType.FLOAT
ram_domain = ram_feature.float_domain
ram_domain.min = 1.0
ram_domain.max = 64.0

In [35]:
# Define Memory as string
memory_feature = schema.feature.add()
memory_feature.name = 'Memory'
memory_feature.type = schema_pb2.FeatureType.BYTES

In [36]:
# Define OpSys as string
opsys_feature = schema.feature.add()
opsys_feature.name = 'OpSys'
opsys_feature.type = schema_pb2.FeatureType.BYTES

In [37]:
# Define Weight as string
weight_feature = schema.feature.add()
weight_feature.name = 'Weight'
weight_feature.type = schema_pb2.FeatureType.FLOAT

In [38]:
# Define Cpu as string
cpu_feature = schema.feature.add()
cpu_feature.name = 'Cpu'
cpu_feature.type = schema_pb2.FeatureType.BYTES

In [39]:
# Define Gpu as string
gpu_feature = schema.feature.add()
gpu_feature.name = 'Gpu'
gpu_feature.type = schema_pb2.FeatureType.BYTES

In [40]:
tfdv.write_schema_text(schema, 'manual_schema.pbtxt')

In [41]:
# Read manual schema
schema = tfdv.load_schema_text('manual_schema.pbtxt')
schema

feature {
  name: "Price"
  type: FLOAT
  float_domain {
    min: 10000.0
    max: 300000.0
  }
}
feature {
  name: "Company"
  type: BYTES
}
feature {
  name: "TypeName"
  type: BYTES
}
feature {
  name: "Inches"
  type: FLOAT
  float_domain {
    min: 10.0
    max: 32.0
  }
}
feature {
  name: "ScreenResolution"
  type: BYTES
}
feature {
  name: "Ram"
  type: FLOAT
  float_domain {
    min: 1.0
    max: 64.0
  }
}
feature {
  name: "Memory"
  type: BYTES
}
feature {
  name: "OpSys"
  type: BYTES
}
feature {
  name: "Weight"
  type: FLOAT
}
feature {
  name: "Cpu"
  type: BYTES
}
feature {
  name: "Gpu"
  type: BYTES
}

In [44]:
# 2️⃣ Generate statistics for each split
train_stats = tfdv.generate_statistics_from_dataframe(train_df)
eval_stats = tfdv.generate_statistics_from_dataframe(eval_df)
test_stats = tfdv.generate_statistics_from_dataframe(test_df)

In [45]:
# 3️⃣ Validate each split against the manual schema
train_anomalies = tfdv.validate_statistics(train_stats, schema)
eval_anomalies = tfdv.validate_statistics(eval_stats, schema)
test_anomalies = tfdv.validate_statistics(test_stats, schema)

In [46]:
# 4️⃣ Display the anomalies
print('🚀 Train anomalies:')
tfdv.display_anomalies(train_anomalies)

🚀 Train anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)
'__index_level_0__',New column,New column (column in data but not in schema)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)


In [47]:
print('🚀 Eval anomalies:')
tfdv.display_anomalies(eval_anomalies)

🚀 Eval anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)
'__index_level_0__',New column,New column (column in data but not in schema)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)


In [48]:
print('🚀 Test anomalies:')
tfdv.display_anomalies(test_anomalies)

🚀 Test anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'__index_level_0__',New column,New column (column in data but not in schema)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)


In [54]:
if train_anomalies.anomaly_info:
    print(train_anomalies.anomaly_info)
    for feature_name, anomaly_info in train_anomalies.anomaly_info.items():
        desc = anomaly_info.description.lower()
        print(desc)

{'Price': description: "Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)"
severity: ERROR
short_description: "Multiple errors"
reason {
  type: FLOAT_TYPE_SMALL_FLOAT
  short_description: "Out-of-range values"
  description: "Unexpectedly low values: -500<10000(upto six significant digits)"
}
reason {
  type: FLOAT_TYPE_BIG_FLOAT
  short_description: "Out-of-range values"
  description: "Unexpectedly high value: 999999>300000(upto six significant digits)"
}
path {
  step: "Price"
}
, '__index_level_0__': description: "New column (column in data but not in schema)"
severity: ERROR
short_description: "New column"
reason {
  type: SCHEMA_NEW_COLUMN
  short_description: "New column"
  description: "New column (column in data but not in schema)"
}
path {
  step: "__index_level_0__"
}
, 'Inches': description: "Unexpectedly high value: 35.6>32(upto six significant digits)"
severity: ERROR
short_description: "O

In [51]:
json.loads(train_anomalies.anomaly_info)

TypeError: the JSON object must be str, bytes or bytearray, not MessageMapContainer